# E4 — Pipeline demo: de nube rota a STL imprimible

Notebook de demostración para el TFM. Selecciona automáticamente los **N objetos con mejor CD** del test set y muestra para cada uno el pipeline completo:

```
Nube rota (entrada)  →  PoinTr (reconstrucción)  →  STL (malla imprimible)
```

**Visualizaciones por objeto:**
- Panel 1: nube de puntos rota (rojo) + nube completa PoinTr (verde)
- Panel 2: malla STL en sólido con coloring por altura
- Panel 3: malla STL en wireframe (triángulos visibles)

**Selección de objetos:** se evalúa el test set completo con Chamfer Distance y se toman los N mejores (CD más bajo = mejor reconstrucción).

---
## Sección 1 — Setup

In [ ]:
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
import subprocess
from getpass import getpass

for pkg in ['trimesh', 'manifold3d', 'pymeshlab', 'plotly', 'numpy', 'easydict']:
    r = subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "WARN"}] {pkg}')

REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub (ghp_...): ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO_DIR, '-q'], capture_output=True)
    del token; print('Repo clonado.')
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','-q'], capture_output=True)
    print('[OK] Repo actualizado.')

if not os.path.exists('/content/PoinTr'):
    subprocess.run(['git','clone','https://github.com/yuxumin/PoinTr',
                    '/content/PoinTr','--depth=1','-q'], capture_output=True)
    print('PoinTr clonado.')

os.chdir(REPO_DIR)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)

---
## Sección 2 — Configuración

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURACION — solo cambia esta sección
# ══════════════════════════════════════════════════════════════

VERSION_E3    = 'v6_obj_sn'  # modelo PoinTr a usar
N_MEJORES     = 6            # cuántos objetos mostrar (los N con menor CD)
TAMANO_MM     = 100.0        # tamaño de impresión en mm (lado mayor)
POISSON_DEPTH = 10           # profundidad Poisson

# Filtro de outliers en la nube de PoinTr (antes de Poisson)
OUTLIER_FILTER    = True     # True = activar
OUTLIER_K         = 20       # nº vecinos
OUTLIER_STD_RATIO = 1.5      # umbral (menor = más agresivo)

# Guardar figuras HTML para el TFM
GUARDAR_FIGURAS = True
DIR_FIGURAS     = 'E4/figuras_demo'

# Rutas
DRIVE    = '/content/drive/MyDrive'
BASE_E3  = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
BASE_GEN = f'{DRIVE}/Datos_E2_E3/General'

_VERSION_DATASETS = {
    'v5_obj':    ['obj'], 'v5_fb_obj': ['fb','obj'],
    'v6_obj_sn': ['obj','sn'], 'v6_all': ['fb','obj','sn'],
}
_partes = _VERSION_DATASETS.get(VERSION_E3, ['obj'])

from pathlib import Path
Path(DIR_FIGURAS).mkdir(parents=True, exist_ok=True)

print(f'Modelo : {VERSION_E3}')
print(f'Mostrar: top {N_MEJORES} por CD')
print(f'Outlier: {"ON" if OUTLIER_FILTER else "OFF"} (K={OUTLIER_K}, ratio={OUTLIER_STD_RATIO})')

# Galería de previsualización
N_PREVIEW = 20   # cuántos objetos mostrar en la galería (los mejores por CD)
                 # Sube este número si quieres ver más candidatos antes de elegir

---
## Sección 3 — Cargar modelo PoinTr + dataset

In [ ]:
import sys, types, glob as _glob, random, importlib
import torch, torch.nn as nn, numpy as np
from pathlib import Path
from easydict import EasyDict

# ── Sys path ────────────────────────────────────────────────
for p in ['/content/TFM', '/content/PoinTr']:
    if p in sys.path: sys.path.remove(p)
sys.path.insert(0, '/content/TFM')
sys.path.insert(0, '/content/PoinTr')
importlib.invalidate_caches()
os.chdir('/content/TFM')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'GPU: {torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU"}')

# ── Reset + mocks PoinTr ─────────────────────────────────────
_pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
_del = [k for k,v in sys.modules.items()
        if any(k==p or k.startswith(p+'.') for p in _pfx)
        or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
for k in _del: del sys.modules[k]
importlib.invalidate_caches()

for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
    try:
        src=open(fp,encoding='utf-8').read(); new=src.replace('.cuda()',f'.to("{DEVICE_STR}")')
        if new!=src: open(fp,'w',encoding='utf-8').write(new)
    except: pass

def _force(n,a):
    m=types.ModuleType(n)
    for k,v in a.items(): setattr(m,k,v)
    sys.modules[n]=m
def _inject(n,a):
    if n not in sys.modules: _force(n,a)
def _cr(a,b): d=torch.cdist(a,b,p=2); return d.min(2).values,d.min(1).values
class _CL1(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
_ch={'ChamferDistanceL1':_CL1,'ChamferDistanceL2':_CL1,'ChamferDistanceL1_PM':_CL1,'chamfer_3DDist':_cr}
for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']: _force(n,_ch)
if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz,np_):
        B,N,_=xyz.shape; dev=xyz.device
        idx=torch.zeros(B,np_,dtype=torch.int32,device=dev); dist=torch.full((B,N),1e10,device=dev)
        far=torch.randint(0,N,(B,),dtype=torch.long,device=dev); bi=torch.arange(B,dtype=torch.long,device=dev)
        for i in range(np_):
            idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1)
            dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
        return idx
    def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
    def _bq(r,ns,xyz,nxyz):
        d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
        return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
    def _grp(f,idx):
        B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
        return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
    def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
    def _3i(f,idx,w):
        B,C,M=f.shape; N=idx.shape[1]
        return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
    _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
    _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
    sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu
class _KNN(nn.Module):
    def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
    def forward(self,ref,query):
        if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
        r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
        d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
        return dk.transpose(1,2),ik.transpose(1,2)
_force('knn_cuda',{'KNN':_KNN})
def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
class _Emd(nn.Module):
    def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                   ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                   ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                   ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
    for pfx in ['','extensions.']: _inject(pfx+base,attrs)

# ── Cargar modelo ────────────────────────────────────────────
ckpt_path = f'E3/checkpoints_pointr_{VERSION_E3}/best.pt'
if not Path(ckpt_path).exists():
    ckpt_path = f'{BASE_E3}/modelos/{VERSION_E3}/best.pt'
ck = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Modelo PoinTr {VERSION_E3} — best epoch {ck["epoch"]}')
try:
    from models.build import build_model_from_cfg; model=build_model_from_cfg(EasyDict(ck['model_cfg']))
except:
    from models.PoinTr import PoinTr; model=PoinTr(EasyDict(ck['model_cfg']))
model.load_state_dict(ck['model_state_dict']); model=model.to(device).eval()
print(f'Parámetros: {sum(p.numel() for p in model.parameters()):,}')

# ── Dataset — test set ───────────────────────────────────────
import E3.dataset as _ds; _ds.CENTRAR_EN_ROTO=False
from E3.dataset import construir_pares, ShapeCompletionDataset

FUENTES = {'obj':(f'{BASE_GEN}/roturas_Objaverse_v2','Datos/objaverse/roturas_v2'),
           'sn': (f'{BASE_GEN}/shapenet_roturas','Datos/shapenet/roturas'),
           'fb': (f'{BASE_GEN}/Fantastik_Break_Procesado_v2','Datos/fantastic_breaks/procesado_v2')}
carpetas = [v[1] for k,v in FUENTES.items() if k in _partes and Path(v[1]).exists()]
if not carpetas:
    carpetas = [v[0] for k,v in FUENTES.items() if k in _partes and Path(v[0]).exists()]

todos = construir_pares(carpetas)
rng = random.Random(42); rng.shuffle(todos)
n = len(todos); nt=int(0.8*n); nv=int(0.1*n)
TEST = todos[nt+nv:]
print(f'Test set: {len(TEST)} pares')

---
## Sección 4 — Evaluar test set completo y seleccionar los N mejores

Calcula CD para todos los pares del test set y se queda con los N con menor CD.
Esto garantiza que vemos objetos donde PoinTr funciona bien.

**Puede tardar 2-5 minutos** dependiendo del tamaño del test set.

In [ ]:
import torch, numpy as np
from pathlib import Path

def inferir(roto_np):
    with torch.no_grad():
        inp = torch.tensor(roto_np).unsqueeze(0).to(device)
        out = model(inp)
        return (out[-1] if isinstance(out,(list,tuple)) else out).squeeze(0).cpu().numpy()

def cd_numpy(p, g):
    pt=torch.tensor(p).unsqueeze(0); gt=torch.tensor(g).unsqueeze(0)
    d=torch.cdist(pt,gt,p=2)
    return ((d.min(2).values.mean()+d.min(1).values.mean())/2).item()

def fscore_numpy(p, g, th=0.01):
    pt=torch.tensor(p).unsqueeze(0); gt=torch.tensor(g).unsqueeze(0)
    dpg=torch.cdist(pt,gt,p=2); dgp=torch.cdist(gt,pt,p=2)
    pr=(dpg.min(2).values<th).float().mean(); re=(dgp.min(2).values<th).float().mean()
    return 0.0 if pr+re<1e-8 else (2*pr*re/(pr+re)).item()

def filtrar_outliers(pred):
    if not OUTLIER_FILTER or len(pred) <= OUTLIER_K + 1:
        return pred, 0
    from scipy.spatial import cKDTree
    tree = cKDTree(pred)
    dists, _ = tree.query(pred, k=OUTLIER_K + 1)
    mean_d = dists[:, 1:].mean(axis=1)
    umbral = mean_d.mean() + OUTLIER_STD_RATIO * mean_d.std()
    mask = mean_d < umbral
    return pred[mask], int((~mask).sum())

print(f'Evaluando {len(TEST)} pares...')
resultados = []

for i, (ruta_r, ruta_c) in enumerate(TEST):
    roto = np.load(ruta_r).astype(np.float32)
    comp = np.load(ruta_c).astype(np.float32)
    pred = inferir(roto)
    pred_filt, n_elim = filtrar_outliers(pred)
    cd = cd_numpy(pred_filt, comp)
    fs = fscore_numpy(pred_filt, comp)
    nombre = Path(ruta_r).stem.replace('_roto','')
    resultados.append({'nombre': nombre, 'ruta_r': ruta_r, 'ruta_c': ruta_c,
                       'cd': cd, 'fs': fs, 'pred': pred_filt,
                       'roto': roto, 'comp': comp, 'n_elim': n_elim})
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(TEST)}...')

resultados.sort(key=lambda x: x['cd'])
SELECCIONADOS = resultados[:N_MEJORES]

print(f'\nTop {N_MEJORES} objetos (menor CD = mejor reconstrucción):')
print(f'{"#":<3} {"Nombre":<50} {"CD":>7} {"F-Score":>8} {"Outliers":>9}')
print('-'*80)
for i, r in enumerate(SELECCIONADOS):
    print(f'{i+1:<3} {r["nombre"][:49]:<50} {r["cd"]:>7.4f} {r["fs"]:>8.4f} {r["n_elim"]:>9}')

print(f'\nCD medio test set: {np.mean([r["cd"] for r in resultados]):.4f}')
print(f'CD peor (top 5 peores):')
for r in resultados[-5:]:
    print(f'  {r["nombre"][:50]}  CD={r["cd"]:.4f}')

---
## Sección 5a — Galería de candidatos

Muestra los **N_PREVIEW** mejores objetos por CD como nubes de puntos interactivas.
Rota cada figura con el ratón, evalúa si la reconstrucción tiene forma reconocible,
y **apunta los números** que quieres procesar en la celda siguiente.

In [ ]:
import plotly.graph_objects as go
import numpy as np

_esc = dict(
    xaxis=dict(showticklabels=False, title='', backgroundcolor='#111',
               gridcolor='#333', zerolinecolor='#333'),
    yaxis=dict(showticklabels=False, title='', backgroundcolor='#111',
               gridcolor='#333', zerolinecolor='#333'),
    zaxis=dict(showticklabels=False, title='', backgroundcolor='#111',
               gridcolor='#333', zerolinecolor='#333'),
    bgcolor='#111', aspectmode='data'
)

candidatos = resultados[:N_PREVIEW]
print(f'Galería — top {len(candidatos)} objetos por CD')
print(f'Rota cada figura con el ratón. Apunta los números que te gustan.')
print(f'{"#":<4} {"Nombre":<52} {"CD":>7} {"F-Score":>8}')
print('-'*75)
for i, r in enumerate(candidatos):
    print(f'{i+1:<4} {r["nombre"][:51]:<52} {r["cd"]:>7.4f} {r["fs"]:>8.4f}')

print()
for i, r in enumerate(candidatos):
    roto = r['roto']
    pred = r['pred']
    comp = r['comp']

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=comp[:,0], y=comp[:,1], z=comp[:,2], mode='markers',
        name='GT', marker=dict(size=1.2, color='#42A5F5', opacity=0.08)))
    fig.add_trace(go.Scatter3d(
        x=pred[:,0], y=pred[:,1], z=pred[:,2], mode='markers',
        name='PoinTr', marker=dict(size=2, color='#66BB6A', opacity=0.85)))
    fig.add_trace(go.Scatter3d(
        x=roto[:,0], y=roto[:,1], z=roto[:,2], mode='markers',
        name='Roto', marker=dict(size=3, color='#EF5350', opacity=0.95)))

    fig.update_layout(
        scene=_esc,
        title=dict(
            text=f'<b>#{i+1}  {r["nombre"]}</b>'
                 f'  CD={r["cd"]:.4f}  F={r["fs"]:.4f}<br>'
                 f'<sup>🔴 Roto (entrada) | 🟢 PoinTr (reconstrucción) | 🔵 GT</sup>',
            font=dict(color='white', size=10), x=0.5),
        legend=dict(font=dict(color='white'), bgcolor='rgba(0,0,0,0.5)',
                    orientation='h', y=-0.02),
        paper_bgcolor='#111',
        height=480, width=600,
        margin=dict(l=0, r=0, t=65, b=30))
    fig.show()


---
## Sección 5b — Selección manual

Mira la galería de arriba y elige los objetos que quieres procesar.

- **Opción A** — por número de la galería: `SELECCION_NUMEROS = [1, 3, 7]`
- **Opción B** — por nombre exacto: copia el nombre de la tabla y ponlo en `SELECCION_NOMBRES`
- **Opción C** — automático: deja ambas listas vacías → usa el top N_MEJORES del config

In [ ]:
# ── Sección 5b: Selección interactiva ────────────────────────────────────────
# Mira la galería de arriba y escribe los números que quieres.
# Puedes poner uno o varios separados por comas: 1,3,7
# Si dejas en blanco y pulsas Enter → se usan automáticamente los top N_MEJORES.

print('=' * 60)
print('SELECCION DE OBJETOS')
print('=' * 60)
print(f'Disponibles: {len(resultados[:N_PREVIEW])} objetos en la galería')
print(f'Escribe los números separados por comas (ej: 1,3,5)')
print(f'O pulsa Enter sin escribir nada para usar el top {N_MEJORES} automático')
print()

entrada = input('Números a procesar: ').strip()

if entrada:
    try:
        numeros = [int(x.strip()) for x in entrada.split(',') if x.strip()]
        numeros_validos = [n for n in numeros if 1 <= n <= len(resultados)]
        invalidos = [n for n in numeros if n not in numeros_validos]
        if invalidos:
            print(f'  ⚠️  Números fuera de rango ignorados: {invalidos}')
        SELECCIONADOS = [resultados[n-1] for n in numeros_validos]
        modo = f'manual — números: {numeros_validos}'
    except ValueError:
        print('  ⚠️  Entrada no válida — usando top automático')
        SELECCIONADOS = resultados[:N_MEJORES]
        modo = f'automático (error de entrada)'
else:
    SELECCIONADOS = resultados[:N_MEJORES]
    modo = f'automático — top {N_MEJORES} por CD'

print()
print(f'Modo: {modo}')
print(f'Objetos seleccionados ({len(SELECCIONADOS)}):')
print()
print(f'  {"#":<4} {"Nombre":<52} {"CD":>7} {"F-Score":>8}')
print('  ' + '-'*73)
for i, r in enumerate(SELECCIONADOS):
    print(f'  {i+1:<4} {r["nombre"][:51]:<52} {r["cd"]:>7.4f} {r["fs"]:>8.4f}')

print()
if SELECCIONADOS:
    print(f'✅ Selección guardada. Ejecuta la Sección 5 para generar los STLs.')
else:
    print('⚠️  Sin objetos seleccionados — revisa los números e intenta de nuevo.')


---
## Sección 5 — Generar STL para los objetos seleccionados

In [ ]:
import pymeshlab, trimesh, numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

# ══════════════════════════════════════════════════════════════════
# SUAVIZADO DE NUBE — reduce ruido sin añadir puntos fantasma
# ══════════════════════════════════════════════════════════════════
def suavizar_nube(pts, k=15, iters=3):
    """
    Laplacian smoothing sobre la nube de puntos:
    cada punto se mueve hacia el centroide de sus K vecinos.

    Por qué funciona: PoinTr genera algunos puntos con ruido/scatter
    (especialmente en las zonas donde la forma era incierta).
    El suavizado aplana ese ruido sin cambiar la topología global.
    Esto hace que BPA y Poisson generen menos geometría interna falsa.

    Importante: NO añade puntos nuevos (a diferencia de densificar),
    así que no empeora el problema de los blobs.
    """
    tree = cKDTree(pts)
    _, idx = tree.query(pts, k=k+1)
    result = pts.copy()
    for _ in range(iters):
        result = result[idx[:, 1:]].mean(axis=1)  # media de K vecinos
    return result.astype(np.float32)

# ══════════════════════════════════════════════════════════════════
# BPA — Ball Pivoting Algorithm
# ══════════════════════════════════════════════════════════════════
def intentar_bpa(pts, nombre, min_caras=500):
    """
    Prueba BPA con radio automático (pymeshlab elige) y con radios
    calculados desde la densidad real de la nube.

    BPA no inventa geometría: solo crea triángulos entre puntos
    que la bola puede tocar simultáneamente. Donde no hay puntos,
    deja un hueco. Para el TFM esto es mejor que Poisson, porque
    la forma es reconocible aunque incompleta.
    """
    bbox_diag = float(np.linalg.norm(pts.max(axis=0) - pts.min(axis=0)))
    if bbox_diag < 1e-6:
        return None, 0

    tree = cKDTree(pts)
    dists, _ = tree.query(pts, k=2)           # vecino MAS CERCANO (no el k=11)
    dist_1nn = float(dists[:, 1].mean())       # ~0.056 para 4000 pts en esfera unitaria

    # Radio correcto: 1.5x distancia al 1er vecino (~0.084 abs → ~2.4% del bbox)
    # Antes usabamos k=11 → radio ~3x mayor → BPA no encontraba triangulos
    radios_pct = [0, (dist_1nn * 1.5 / bbox_diag) * 100,
                      (dist_1nn * 3.0 / bbox_diag) * 100]
    mejor_mesh, mejor_caras = None, 0

    for pct in radios_pct:
        try:
            ms = pymeshlab.MeshSet()
            ms.add_mesh(pymeshlab.Mesh(vertex_matrix=pts.astype(np.float64)))
            ms.compute_normal_for_point_clouds(k=12, smoothiter=3)
            ms.generate_surface_reconstruction_ball_pivoting(
                ballradius=pymeshlab.Percentage(pct),
                clustering=0.2, creaseangle=90.0, deletefaces=False)
            m = ms.current_mesh()
            n = len(m.face_matrix())
            if n > mejor_caras:
                mejor_mesh = trimesh.Trimesh(vertices=m.vertex_matrix(),
                                             faces=m.face_matrix(), process=False)
                mejor_caras = n
        except Exception:
            pass

    if mejor_caras >= min_caras:
        return mejor_mesh, mejor_caras
    return None, 0

# ══════════════════════════════════════════════════════════════════
# POISSON fallback — depth=7 (menos fantasmas que depth=8 o 10)
# ══════════════════════════════════════════════════════════════════
def intentar_poisson(pts, nombre, depth=7):
    """
    depth=7 (antes usábamos 8 o 10):
    - depth=10 con 4000 pts → ~15 triángulos/punto → 60k caras → muchos fantasmas
    - depth=8  con 4000 pts → ~11 triángulos/punto → 45k caras → algunos fantasmas
    - depth=7  con 4000 pts → ~5 triángulos/punto  → 20k caras → menos fantasmas
    El resultado es más basto pero más honesto con la forma real.
    """
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=pts.astype(np.float64)), nombre)
    ms.compute_normal_for_point_clouds(k=20, smoothiter=2)
    ms.generate_surface_reconstruction_screened_poisson(depth=depth, scale=1.1)
    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=200)
    ms.apply_coord_laplacian_smoothing(stepsmoothnum=2)
    m = ms.current_mesh()
    return trimesh.Trimesh(vertices=m.vertex_matrix(), faces=m.face_matrix(), process=False)

# ══════════════════════════════════════════════════════════════════
# REPARACIÓN topológica
# ══════════════════════════════════════════════════════════════════
def reparar_malla(mesh):
    reps = []
    comps = mesh.split(only_watertight=False)
    if len(comps) > 1:
        mesh = max(comps, key=lambda c: len(c.faces))
        reps.append(f'comp({len(comps)}→1)')
    try:
        import manifold3d
        m = manifold3d.Manifold(manifold3d.Mesh(
            vert_properties=np.array(mesh.vertices, dtype=np.float32),
            tri_verts=np.array(mesh.faces, dtype=np.uint32)))
        out = m.to_mesh()
        res = trimesh.Trimesh(vertices=np.array(out.vert_properties),
                              faces=np.array(out.tri_verts), process=False)
        if len(res.vertices) > 0:
            mesh = res; reps.append('manifold3d')
    except Exception:
        try:
            ms2 = pymeshlab.MeshSet()
            ms2.add_mesh(pymeshlab.Mesh(
                vertex_matrix=np.array(mesh.vertices, dtype=np.float64),
                face_matrix=np.array(mesh.faces, dtype=np.int32)))
            ms2.meshing_remove_duplicate_faces()
            ms2.meshing_remove_null_faces()
            try: ms2.meshing_repair_non_manifold_edges(method=0)
            except Exception: pass
            for max_h in [30, 150, 500]:
                try: ms2.meshing_close_holes(maxholesize=max_h)
                except Exception: pass
            m2 = ms2.current_mesh()
            res2 = trimesh.Trimesh(vertices=m2.vertex_matrix(),
                                   faces=m2.face_matrix(), process=False)
            if len(res2.vertices) > 0:
                mesh = res2; reps.append('pymeshlab')
        except Exception: pass
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fix_winding(mesh)
    mask = mesh.nondegenerate_faces()
    if (~mask).sum() > 0: mesh.update_faces(mask)
    mesh.process(validate=False)
    return mesh, reps

# ══════════════════════════════════════════════════════════════════
# PIPELINE PRINCIPAL
# ══════════════════════════════════════════════════════════════════
def pred_a_stl(pred_orig, nombre):
    """
    Pipeline E3→E4:
      1. Suavizar nube (Laplacian, k=15, 3 iter) — reduce ruido/scatter de PoinTr
      2. BPA auto + radios calculados — no inventa geometría
      3. Si BPA falla → Poisson depth=7 (menos fantasmas que depth=8/10)
      4. Reparación topológica + escalar a TAMANO_MM
    """
    Path('E4/stl_demo').mkdir(parents=True, exist_ok=True)

    n_orig = len(pred_orig)

    # 1. Suavizar (NO densificar — densificar+Poisson empeora los fantasmas)
    pred_suav = suavizar_nube(pred_orig, k=15, iters=3)
    print(f'    Suavizado: {n_orig} pts (Laplacian k=15, 3 iter)')

    # 2. BPA sobre nube suavizada
    mesh_bpa, n_bpa = intentar_bpa(pred_suav, nombre)
    if mesh_bpa is not None:
        print(f'    [BPA ✓] {n_bpa} caras')
        mesh_raw = mesh_bpa
        metodo = 'BPA+suavizado'
    else:
        # 3. Poisson depth=7 (menos fantasmas)
        print(f'    [BPA ✗] → Poisson depth=7')
        mesh_raw = intentar_poisson(pred_suav, nombre, depth=7)
        metodo = 'Poisson_d7+suavizado'
        print(f'    [Poisson d7] {len(mesh_raw.faces)} caras')

    # 4. Escalar
    mesh_raw.apply_translation(-mesh_raw.centroid)
    lado = mesh_raw.bounding_box.extents.max()
    if lado > 0: mesh_raw.apply_scale(TAMANO_MM / lado)

    # 5. Reparar
    mesh_rep, reps = reparar_malla(mesh_raw)
    wt = bool(mesh_rep.is_watertight)
    eu = int(mesh_rep.euler_number)
    return mesh_rep, wt, eu, metodo + (' | ' + ' | '.join(reps) if reps else '')


print('Generando STLs: suavizado Laplacian + BPA auto + Poisson depth=7')
print('─' * 60)

for r in SELECCIONADOS:
    print(f'\n  {r["nombre"][:55]}')
    mesh, wt, eu, reps_str = pred_a_stl(r['pred'], r['nombre'])
    r['mesh'] = mesh
    r['watertight'] = wt
    r['euler'] = eu
    r['reps'] = reps_str
    print(f'    → {"watertight ✓" if wt else "NO watertight"}  euler={eu}  caras={len(mesh.faces):,}')
    print(f'    → {reps_str}')


---
## Sección 6 — Visualización completa del pipeline

Para cada objeto muestra:
- **Izquierda**: nube rota (rojo) + reconstrucción PoinTr (verde)
- **Centro**: malla STL sólida con coloring por altura
- **Derecha**: malla STL en **wireframe** (triángulos visibles)

Rota con el ratón en cada panel.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from pathlib import Path

_escena_base = dict(
    xaxis=dict(showticklabels=False, title='', backgroundcolor='#111',
               gridcolor='#333', zerolinecolor='#333'),
    yaxis=dict(showticklabels=False, title='', backgroundcolor='#111',
               gridcolor='#333', zerolinecolor='#333'),
    zaxis=dict(showticklabels=False, title='', backgroundcolor='#111',
               gridcolor='#333', zerolinecolor='#333'),
    bgcolor='#111', aspectmode='data'
)

def escena(): return dict(**_escena_base)

def aristas_malla(mesh):
    """Extrae aristas únicas de la malla para el wireframe."""
    verts = np.array(mesh.vertices)
    faces = np.array(mesh.faces)
    # Pares de vértices de cada arista de cada triángulo
    edges = set()
    for f in faces:
        for i in range(3):
            e = tuple(sorted([f[i], f[(i+1)%3]]))
            edges.add(e)
    edges = list(edges)
    # Construir arrays x,y,z con None como separador entre aristas
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [verts[a,0], verts[b,0], None]
        ys += [verts[a,1], verts[b,1], None]
        zs += [verts[a,2], verts[b,2], None]
    return xs, ys, zs

for idx, r in enumerate(SELECCIONADOS):
    nombre = r['nombre']
    roto   = r['roto']
    pred   = r['pred']
    comp   = r['comp']
    mesh   = r['mesh']
    cd_val = r['cd']
    fs_val = r['fs']
    wt_str = 'APTO' if r['watertight'] else 'NO watertight'

    verts = np.array(mesh.vertices)
    faces = np.array(mesh.faces)

    # ── Panel 1: Nube rota + PoinTr ─────────────────────────
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter3d(
        x=comp[:,0], y=comp[:,1], z=comp[:,2], mode='markers',
        name='GT completo', marker=dict(size=1.5, color='#42A5F5', opacity=0.10)))
    fig1.add_trace(go.Scatter3d(
        x=pred[:,0], y=pred[:,1], z=pred[:,2], mode='markers',
        name='PoinTr (reconstrucción)', marker=dict(size=2, color='#66BB6A', opacity=0.85)))
    fig1.add_trace(go.Scatter3d(
        x=roto[:,0], y=roto[:,1], z=roto[:,2], mode='markers',
        name='Roto (entrada)', marker=dict(size=3, color='#EF5350', opacity=0.95)))
    fig1.update_layout(
        scene=escena(),
        title=dict(text=f'<b>[{idx+1}] {nombre}</b>  CD={cd_val:.4f}  F={fs_val:.4f}<br>'
                        f'<sup>Rojo=roto entrada | Verde=PoinTr | Azul=GT completo</sup>',
                   font=dict(color='white', size=11), x=0.5),
        legend=dict(font=dict(color='white'), bgcolor='rgba(0,0,0,0.5)'),
        paper_bgcolor='#111', height=550, width=650,
        margin=dict(l=0,r=0,t=65,b=0))
    fig1.show()

    if len(faces) == 0:
        print(f'  [{nombre}] malla vacía — sin visualización 3D')
        continue

    # ── Panel 2: STL sólido ──────────────────────────────────
    z = verts[:,2]
    intensidad = (z - z.min()) / (np.ptp(z) + 1e-8)
    fig2 = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intensidad,
        colorscale=[[0,'#0D47A1'],[0.5,'#29B6F6'],[1,'#E1F5FE']],
        showscale=False,
        lighting=dict(ambient=0.3, diffuse=0.85, roughness=0.3, specular=0.6),
        lightposition=dict(x=200, y=300, z=400)
    ))
    fig2.update_layout(
        scene=escena(),
        title=dict(text=f'<b>STL sólido</b> [{wt_str}]  euler={r["euler"]}  '
                        f'{len(faces):,} caras',
                   font=dict(color='white', size=11), x=0.5),
        paper_bgcolor='#111', font=dict(color='white'),
        height=550, width=650, margin=dict(l=0,r=0,t=50,b=0))
    fig2.show()

    # ── Panel 3: STL wireframe ────────────────────────────────
    # Para objetos grandes (>50k caras) mostramos solo 1 de cada N aristas
    # para que Plotly no se quede sin memoria.
    MAX_ARISTAS = 30000
    xs, ys, zs = aristas_malla(mesh)
    n_aristas = len([x for x in xs if x is None])
    if n_aristas > MAX_ARISTAS:
        paso = n_aristas // MAX_ARISTAS + 1
        # Filtrar: tomar 1 de cada 'paso' grupos de 3 elementos (cada arista = 3 slots)
        triplets = [(xs[i*3:i*3+3], ys[i*3:i*3+3], zs[i*3:i*3+3])
                    for i in range(n_aristas) if i % paso == 0]
        xs = [v for t in triplets for v in t[0]]
        ys = [v for t in triplets for v in t[1]]
        zs = [v for t in triplets for v in t[2]]
        print(f'  Wireframe: {n_aristas:,} aristas → mostrando {len(triplets):,} (1/{paso})')

    fig3 = go.Figure()
    # Relleno semitransparente
    fig3.add_trace(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        color='#29B6F6', opacity=0.12, showscale=False))
    # Aristas
    fig3.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='lines',
        line=dict(color='#80DEEA', width=1), name='Aristas',
        hoverinfo='none'))
    fig3.update_layout(
        scene=escena(),
        title=dict(text=f'<b>STL wireframe</b> (triángulos visibles)  '
                        f'{len(faces):,} caras',
                   font=dict(color='white', size=11), x=0.5),
        showlegend=False,
        paper_bgcolor='#111', font=dict(color='white'),
        height=550, width=650, margin=dict(l=0,r=0,t=50,b=0))
    fig3.show()

    # Guardar como HTML
    if GUARDAR_FIGURAS:
        fig1.write_html(f'{DIR_FIGURAS}/{nombre}_nubes.html')
        fig2.write_html(f'{DIR_FIGURAS}/{nombre}_solido.html')
        fig3.write_html(f'{DIR_FIGURAS}/{nombre}_wireframe.html')

    print(f'[{idx+1}] {nombre[:50]}  CD={cd_val:.4f}  watertight={r["watertight"]}  caras={len(faces):,}')
    print()

---
## Sección 7 — Tabla resumen

Tabla comparativa de los objetos seleccionados con sus métricas.

In [ ]:
import plotly.graph_objects as go

cols = {
    'Objeto':       [r['nombre'][:40] for r in SELECCIONADOS],
    'CD-L1 ↓':      [f'{r["cd"]:.4f}' for r in SELECCIONADOS],
    'F-Score ↑':    [f'{r["fs"]:.4f}' for r in SELECCIONADOS],
    'Outliers':     [str(r['n_elim']) for r in SELECCIONADOS],
    'Watertight':   ['✅ SÍ' if r['watertight'] else '❌ NO' for r in SELECCIONADOS],
    'Euler':        [str(r['euler']) for r in SELECCIONADOS],
    'Reparaciones': [r['reps'][:50] for r in SELECCIONADOS],
}

colores = ['#C8E6C9' if r['watertight'] else '#FFCDD2' for r in SELECCIONADOS]

fig = go.Figure(go.Table(
    header=dict(
        values=[f'<b>{c}</b>' for c in cols.keys()],
        fill_color='#1565C0', font=dict(color='white', size=11),
        align='center', height=28),
    cells=dict(
        values=list(cols.values()),
        fill_color=[colores]*len(cols),
        align=['left','center','center','center','center','center','left'],
        font=dict(size=10), height=24)
))
fig.update_layout(
    title='<b>Top objetos — pipeline E3→E4</b>  (verde = watertight/apto | rojo = no apto)',
    height=80 + 26*len(SELECCIONADOS),
    margin=dict(l=10,r=10,t=50,b=10))
fig.show()

if GUARDAR_FIGURAS:
    fig.write_html(f'{DIR_FIGURAS}/tabla_resumen.html')

aptos = sum(1 for r in SELECCIONADOS if r['watertight'])
print(f'\n{aptos}/{len(SELECCIONADOS)} objetos watertight entre los top {N_MEJORES} por CD')
print(f'CD medio top {N_MEJORES}: {np.mean([r["cd"] for r in SELECCIONADOS]):.4f}')
print(f'CD medio test set : {np.mean([r["cd"] for r in resultados]):.4f}')

---
## Sección 8 — Diagnóstico: E4 con nubes GT (sin pasar por PoinTr)

**Objetivo:** confirmar dónde está el cuello de botella del pipeline.

- Si las nubes GT dan STLs reconocibles → **E4 funciona**, el problema es el ruido/huecos en la salida de PoinTr (E3).
- Si las nubes GT también dan blobs → el problema es BPA/Poisson con 2048 puntos en general.

Las nubes GT (`comp`) son 2048 puntos limpios muestreados directamente del mesh original.
No tienen ruido ni zonas vacías — son la entrada "ideal" a E4.

In [ ]:
# ── Diagnóstico: mismo pipeline E4 pero con nubes GT completas ───────────────
# En lugar de pred (salida de PoinTr), usamos comp (GT completo limpio).
# Esto aísla si el problema es E3 o E4.

print('DIAGNOSTICO: E4 con nubes GT (sin PoinTr)')
print('=' * 60)
print('GT = nubes perfectas muestreadas del mesh original')
print('Si estas dan buenos STLs → el problema esta en la salida de PoinTr')
print()

GT_SELECCIONADOS = resultados[:N_MEJORES]  # los mismos objetos top por CD

for r in GT_SELECCIONADOS:
    print(f'  {r["nombre"][:55]}')
    # OJO: comp en lugar de pred
    mesh, wt, eu, reps_str = pred_a_stl(r['comp'], r['nombre'] + '_GT')
    n_faces = len(mesh.faces) if mesh is not None else 0
    print(f'    → {"watertight [APTO]" if wt else "NO watertight"}  euler={eu}  caras={n_faces:,}')
    print(f'    → {reps_str}')
    print()

# Visualizar el primero como muestra
r0 = GT_SELECCIONADOS[0]
mesh_gt, wt_gt, eu_gt, _ = pred_a_stl(r0['comp'], r0['nombre'] + '_GT_viz')

if mesh_gt is not None and len(mesh_gt.faces) > 0:
    import plotly.graph_objects as go
    verts = mesh_gt.vertices
    faces = mesh_gt.faces
    z = verts[:,2]
    intensidad = (z - z.min()) / (np.ptp(z) + 1e-8)

    fig = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intensidad,
        colorscale=[[0,'#1B5E20'],[0.5,'#66BB6A'],[1,'#E8F5E9']],
        showscale=False,
        lighting=dict(ambient=0.3, diffuse=0.9, roughness=0.2, specular=0.6),
        lightposition=dict(x=200, y=300, z=400)))

    fig.update_layout(
        scene=dict(
            xaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
            yaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
            zaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
            bgcolor='#111', aspectmode='data'),
        title=dict(
            text=f'<b>STL desde GT (no PoinTr)</b> — {r0["nombre"]}<br>'
                 f'<sup>{"WATERTIGHT" if wt_gt else "no watertight"}  euler={eu_gt}  {len(faces):,} caras</sup>',
            font=dict(color='white', size=12), x=0.5),
        paper_bgcolor='#111',
        height=600, width=700,
        margin=dict(l=0,r=0,t=65,b=0))
    fig.show()
    print(f'Interpretación: si esta figura parece una taza → E4 OK, problema en PoinTr.')
    print(f'Si parece un blob → el problema es BPA/Poisson con 2048 puntos en general.')
else:
    print('Malla vacía — revisar E4 pipeline')
